# 4D tracking of segmented DFXM dislocation-cell volumes

This notebook loads the segmentation outputs produced by `segment_111_june_all_notebook_params.py`, expands all volumes onto a common z-layer grid, registers the volumes across strain, extracts the common overlapping region, and performs overlap-based tracking of cell labels.

It is designed for:

```text
~/Documents/Data/4dcells/111_june/disell_batch_output/
```

Expected per-dataset files:

```text
registered_volume.npy
seg_input.npy
mask.npy
labels.npy
rgb_volume.npy
cell_table.csv
neighbour_table.csv
summary.json
```

The notebook deliberately keeps registration simple at first: translation-only registration using phase cross-correlation. That is the right first diagnostic. Do not over-interpret tracking events until the registration overlays look convincing.


## 1. Imports and paths

In [ ]:
from pathlib import Path
import json
import re
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import ndimage as ndi
from scipy import stats
from skimage.registration import phase_cross_correlation
from skimage.segmentation import relabel_sequential

# ---------------------------------------------------------------------
# Main path
# ---------------------------------------------------------------------

output_root = Path("~/Documents/Data/4dcells/111_june/disell_batch_output").expanduser()

dataset_order = [
    "111_cells_2_6-1pct_mosalayers_2x_redo",
    "111_cells_2_6-2pct_mosalayers_2x",
    "111_cells_2_6-3pct_mosalayers_2x",
    "111_cells_2_6-4pct_mosalayers_2x",
    "111_cells_2_6-5pct_mosalayers_2x",
    "111_cells_2_6-7pct_mosalayers_2x",
]

# Nominal labels from folder names. Replace with calibrated strain if needed.
strain_percent = {
    "111_cells_2_6-1pct_mosalayers_2x_redo": 1.0,
    "111_cells_2_6-2pct_mosalayers_2x": 2.0,
    "111_cells_2_6-3pct_mosalayers_2x": 3.0,
    "111_cells_2_6-4pct_mosalayers_2x": 4.0,
    "111_cells_2_6-5pct_mosalayers_2x": 5.0,
    "111_cells_2_6-7pct_mosalayers_2x": 7.0,
}

print("output_root:", output_root)
print("exists:", output_root.exists())

## 2. Load segmented volumes

In [ ]:
datasets = []

for name in dataset_order:
    d = output_root / name
    if not d.exists():
        print("[skip missing]", d)
        continue

    required = [
        "registered_volume.npy",
        "seg_input.npy",
        "mask.npy",
        "labels.npy",
        "rgb_volume.npy",
        "summary.json",
    ]
    missing = [f for f in required if not (d / f).exists()]
    if missing:
        print("[skip incomplete]", name, "missing:", missing)
        continue

    with open(d / "summary.json") as f:
        summary = json.load(f)

    item = {
        "name": name,
        "strain_percent": strain_percent.get(name, summary.get("nominal_strain_percent")),
        "path": d,
        "registered": np.load(d / "registered_volume.npy"),
        "seg_input": np.load(d / "seg_input.npy"),
        "mask": np.load(d / "mask.npy").astype(bool),
        "labels": np.load(d / "labels.npy").astype(np.int32),
        "rgb": np.load(d / "rgb_volume.npy"),
        "summary": summary,
    }

    # Optional tables
    if (d / "cell_table.csv").exists():
        item["cell_table"] = pd.read_csv(d / "cell_table.csv")
    if (d / "neighbour_table.csv").exists():
        item["neighbour_table"] = pd.read_csv(d / "neighbour_table.csv")

    datasets.append(item)

print("loaded:", len(datasets))
for item in datasets:
    print(
        item["name"],
        "strain:", item["strain_percent"],
        "registered:", item["registered"].shape,
        "labels:", item["labels"].shape,
        "max label:", int(item["labels"].max()),
        "coverage:",
        item["summary"].get("final_coverage_inside_mask"),
    )

## 3. Expand all volumes to a common z-layer grid

One dataset may have 12 layers while the others have 11. Do **not** simply crop or blindly pad by one slice. The right thing is to parse the original `layer_names` stored in each `summary.json`, collect all unique acquisition layer IDs, and place each volume into a common z-coordinate stack.

Layers absent in a given strain step are filled with:

```text
registered_volume: NaN
seg_input:         0
mask:              False
labels:            0
rgb:               0
```

The later `common_mask` intersection will automatically keep only voxels/layers present in every strain step.


In [ ]:
def parse_layer_number(layer_name):
    """
    Extract layer number from names like:
        layer_0000_...
        layer_12_...
        layer-12
    """
    m = re.search(r"layer[_-]?(\d+)", layer_name)
    if m is None:
        raise ValueError(f"Could not parse layer number from {layer_name!r}")
    return int(m.group(1))


# Check layer names
for item in datasets:
    layer_names = item["summary"].get("layer_names", None)
    if layer_names is None:
        raise KeyError(f"{item['name']} has no layer_names in summary.json")
    layer_ids = [parse_layer_number(x) for x in layer_names]
    print(item["name"], "n_layers:", len(layer_ids), "layer ids:", layer_ids)


all_layer_ids = sorted(
    {
        parse_layer_number(layer_name)
        for item in datasets
        for layer_name in item["summary"]["layer_names"]
    }
)

layer_to_common_index = {layer_id: i for i, layer_id in enumerate(all_layer_ids)}

print("\ncommon layer IDs:", all_layer_ids)
print("common z size:", len(all_layer_ids))


def expand_z_to_common_layers(arr, layer_names, fill_value):
    """
    Expand a 3D or 4D array from local z-layers into the common z grid.

    arr:
        3D: (Z, Y, X)
        4D: (Z, Y, X, C)

    layer_names:
        list of layer names corresponding to arr's local z axis.
    """
    layer_ids = [parse_layer_number(name) for name in layer_names]

    if arr.shape[0] != len(layer_ids):
        raise ValueError(
            f"z mismatch: arr has {arr.shape[0]} layers, "
            f"but layer_names has {len(layer_ids)} entries"
        )

    out_shape = (len(all_layer_ids),) + arr.shape[1:]
    out = np.full(out_shape, fill_value, dtype=arr.dtype)

    for local_z, layer_id in enumerate(layer_ids):
        common_z = layer_to_common_index[layer_id]
        out[common_z] = arr[local_z]

    return out


# Make expanded copies so raw loaded data remain available if needed.
for item in datasets:
    layer_names = item["summary"]["layer_names"]

    item["registered_commonz"] = expand_z_to_common_layers(
        item["registered"], layer_names, fill_value=np.nan
    )

    item["seg_input_commonz"] = expand_z_to_common_layers(
        item["seg_input"], layer_names, fill_value=0.0
    )

    item["mask_commonz"] = expand_z_to_common_layers(
        item["mask"].astype(bool), layer_names, fill_value=False
    ).astype(bool)

    item["labels_commonz"] = expand_z_to_common_layers(
        item["labels"].astype(np.int32), layer_names, fill_value=0
    ).astype(np.int32)

    item["rgb_commonz"] = expand_z_to_common_layers(
        item["rgb"], layer_names, fill_value=0.0
    )

print("\nExpanded shapes:")
for item in datasets:
    print(
        item["name"],
        "registered:", item["registered_commonz"].shape,
        "labels:", item["labels_commonz"].shape,
        "mask voxels:", np.count_nonzero(item["mask_commonz"]),
    )

## 4. Choose registration feature

The notebook supports several registration features:

```text
mask         : aligns field-of-view/support only. Good first diagnostic.
label_binary : aligns segmented support. Usually similar to mask if coverage is 100%.
channel0     : aligns registered_volume[..., 0].
channel1     : aligns registered_volume[..., 1].
```

Start with `mask`. If `mask`, `channel0`, and `channel1` give similar shifts, tracking is more trustworthy. If they disagree strongly, the tracking events are not reliable yet.


In [ ]:
def robust_standardise(arr, mask=None):
    arr = np.asarray(arr, dtype=np.float32)
    out = arr.copy()

    finite = np.isfinite(out)
    if mask is not None:
        finite &= mask

    if not np.any(finite):
        return np.zeros_like(out, dtype=np.float32)

    med = np.nanmedian(out[finite])
    std = np.nanstd(out[finite])
    out[~np.isfinite(out)] = med
    out = (out - med) / (std + 1e-8)

    if mask is not None:
        out[~mask] = 0.0

    return out.astype(np.float32)


def make_registration_feature(item, mode="mask"):
    if mode == "mask":
        return item["mask_commonz"].astype(np.float32)

    if mode == "label_binary":
        return ((item["labels_commonz"] > 0) & item["mask_commonz"]).astype(np.float32)

    if mode.startswith("channel"):
        c = int(mode.replace("channel", ""))
        arr = item["registered_commonz"][..., c]
        return robust_standardise(arr, mask=item["mask_commonz"])

    raise ValueError(f"Unknown registration mode: {mode}")


registration_mode = "mask"   # try "channel0", "channel1", "label_binary" afterwards

features = [make_registration_feature(item, registration_mode) for item in datasets]

for item, feat in zip(datasets, features):
    print(
        item["name"],
        feat.shape,
        "finite:", np.isfinite(feat).all(),
        "min/max:", float(np.nanmin(feat)), float(np.nanmax(feat)),
    )

## 5. Estimate translation to a common reference

In [ ]:
# Use middle strain as reference.
# With six datasets this is index 3, i.e. usually the 4% dataset.
ref_index = len(datasets) // 2
ref = features[ref_index]

print("reference:", ref_index, datasets[ref_index]["name"], datasets[ref_index]["strain_percent"])

shifts_to_ref = []
errors = []
phasediffs = []

for i, feat in enumerate(features):
    if i == ref_index:
        shift = np.zeros(3, dtype=float)
        error = 0.0
        phasediff = 0.0
    else:
        # shift is the translation to apply to feat to match ref.
        shift, error, phasediff = phase_cross_correlation(
            ref,
            feat,
            upsample_factor=1,       # integer-voxel first pass
            normalization=None,
        )
        shift = np.asarray(shift, dtype=float)

    shifts_to_ref.append(shift)
    errors.append(float(error))
    phasediffs.append(float(phasediff))

registration_table = pd.DataFrame({
    "dataset": [d["name"] for d in datasets],
    "strain_percent": [d["strain_percent"] for d in datasets],
    "shift_z": [s[0] for s in shifts_to_ref],
    "shift_y": [s[1] for s in shifts_to_ref],
    "shift_x": [s[2] for s in shifts_to_ref],
    "registration_error": errors,
    "phasediff": phasediffs,
})

registration_table

## 6. Compare registration modes

This cell estimates shifts using several features. If they agree, good. If not, tracking should be treated as tentative.


In [ ]:
def estimate_shifts_for_mode(mode, ref_index=None):
    feats = [make_registration_feature(item, mode) for item in datasets]
    if ref_index is None:
        ref_index = len(datasets) // 2
    ref = feats[ref_index]

    shifts = []
    errs = []
    for i, feat in enumerate(feats):
        if i == ref_index:
            shifts.append(np.zeros(3, dtype=float))
            errs.append(0.0)
        else:
            shift, err, _ = phase_cross_correlation(
                ref,
                feat,
                upsample_factor=1,
                normalization=None,
            )
            shifts.append(np.asarray(shift, dtype=float))
            errs.append(float(err))
    return shifts, errs


modes_to_compare = ["mask", "label_binary", "channel0", "channel1"]
mode_rows = []

for mode in modes_to_compare:
    shifts, errs = estimate_shifts_for_mode(mode, ref_index=ref_index)
    for item, shift, err in zip(datasets, shifts, errs):
        mode_rows.append({
            "mode": mode,
            "dataset": item["name"],
            "strain_percent": item["strain_percent"],
            "shift_z": shift[0],
            "shift_y": shift[1],
            "shift_x": shift[2],
            "error": err,
        })

mode_comparison = pd.DataFrame(mode_rows)
mode_comparison

## 7. Apply shifts and compute common overlap mask

In [ ]:
def shift_volume(arr, shift, order, cval):
    """
    Shift a 3D or 4D array. For 4D, shift spatial axes only.
    """
    if arr.ndim == 3:
        return ndi.shift(
            arr,
            shift=shift,
            order=order,
            mode="constant",
            cval=cval,
            prefilter=False,
        )

    if arr.ndim == 4:
        out = np.empty_like(arr)
        for c in range(arr.shape[-1]):
            out[..., c] = ndi.shift(
                arr[..., c],
                shift=shift,
                order=order,
                mode="constant",
                cval=cval,
                prefilter=False,
            )
        return out

    raise ValueError(arr.shape)


aligned = []

for item, shift in zip(datasets, shifts_to_ref):
    a = {
        "name": item["name"],
        "strain_percent": item["strain_percent"],
        "shift_to_ref": shift,
        "registered": shift_volume(item["registered_commonz"], shift, order=1, cval=np.nan),
        "seg_input": shift_volume(item["seg_input_commonz"], shift, order=1, cval=0.0),
        "mask": shift_volume(item["mask_commonz"].astype(np.uint8), shift, order=0, cval=0).astype(bool),
        "labels": shift_volume(item["labels_commonz"].astype(np.int32), shift, order=0, cval=0).astype(np.int32),
        "rgb": shift_volume(item["rgb_commonz"], shift, order=1, cval=0.0),
    }
    aligned.append(a)

common_mask = np.logical_and.reduce([a["mask"] for a in aligned])

print("common mask shape:", common_mask.shape)
print("common mask voxels:", np.count_nonzero(common_mask))
print("\ncommon fraction relative to each shifted mask:")
for a in aligned:
    print(
        a["name"],
        f"{np.count_nonzero(common_mask) / max(np.count_nonzero(a['mask']), 1):.4%}",
    )

In [ ]:
def bounding_box(mask, pad=0):
    coords = np.argwhere(mask)
    if coords.size == 0:
        raise ValueError("empty mask")
    lo = np.maximum(coords.min(axis=0) - pad, 0)
    hi = np.minimum(coords.max(axis=0) + pad + 1, mask.shape)
    return tuple(slice(int(l), int(h)) for l, h in zip(lo, hi))


crop = bounding_box(common_mask, pad=0)
print("crop:", crop)

for a in aligned:
    for key in ["registered", "seg_input", "mask", "labels", "rgb"]:
        arr = a[key]
        if arr.ndim == 3:
            a[key + "_crop"] = arr[crop]
        elif arr.ndim == 4:
            a[key + "_crop"] = arr[crop + (slice(None),)]
        else:
            raise ValueError((key, arr.shape))

common_mask_crop = common_mask[crop]

print("cropped common mask:", common_mask_crop.shape)
print("common voxels:", np.count_nonzero(common_mask_crop))

## 8. Visual registration diagnostics

In [ ]:
z = common_mask_crop.shape[0] // 2

fig, axes = plt.subplots(len(aligned), 4, figsize=(18, 3 * len(aligned)))

if len(aligned) == 1:
    axes = axes[None, :]

for row, a in zip(axes, aligned):
    row[0].imshow(a["rgb_crop"][z], aspect=2.8)
    row[0].set_title(f"{a['strain_percent']}% RGB")
    row[0].axis("off")

    row[1].imshow(a["labels_crop"][z], cmap="nipy_spectral", interpolation="nearest", aspect=2.8)
    row[1].set_title("labels")
    row[1].axis("off")

    row[2].imshow(a["mask_crop"][z], cmap="gray", interpolation="nearest", aspect=2.8)
    row[2].set_title("shifted mask")
    row[2].axis("off")

    row[3].imshow(common_mask_crop[z], cmap="gray", interpolation="nearest", aspect=2.8)
    row[3].set_title("common mask")
    row[3].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Overlay each shifted mask boundary/support against the reference mask.
ref_mask = aligned[ref_index]["mask_crop"]

fig, axes = plt.subplots(1, len(aligned), figsize=(4 * len(aligned), 4))

if len(aligned) == 1:
    axes = [axes]

z = common_mask_crop.shape[0] // 2

for ax, a in zip(axes, aligned):
    overlay = np.zeros((*ref_mask[z].shape, 3), dtype=float)
    overlay[..., 0] = ref_mask[z]                         # red = reference mask
    overlay[..., 1] = a["mask_crop"][z]                    # green = this mask
    overlay[..., 2] = common_mask_crop[z] * 0.5             # blue = common overlap

    ax.imshow(overlay, interpolation="nearest", aspect=2.8)
    ax.set_title(f"{a['strain_percent']}%")
    ax.axis("off")

plt.tight_layout()
plt.show()

## 9. Relabel cropped labels and compute coverage

After shifting and cropping, some labels may be truncated by the common-mask crop. This cell sets all voxels outside the common mask to zero and relabels each time step contiguously.


In [ ]:
for a in aligned:
    lab = a["labels_crop"].copy()
    lab[~common_mask_crop] = 0
    lab, _, _ = relabel_sequential(lab)
    a["labels_common"] = np.asarray(lab, dtype=np.int32)

    coverage = np.count_nonzero((a["labels_common"] > 0) & common_mask_crop) / max(np.count_nonzero(common_mask_crop), 1)
    print(a["name"], "labels:", int(a["labels_common"].max()), "coverage:", f"{coverage:.4%}")

## 10. Basic strain evolution summary from cropped common region

In [ ]:
summary_rows = []

for a in aligned:
    labels = a["labels_common"]
    vals, counts = np.unique(labels[(labels > 0) & common_mask_crop], return_counts=True)

    summary_rows.append({
        "dataset": a["name"],
        "strain_percent": a["strain_percent"],
        "n_cells": int(labels.max()),
        "n_valid_voxels": int(np.count_nonzero(common_mask_crop)),
        "median_cell_volume_voxels": float(np.median(counts)) if len(counts) else np.nan,
        "mean_cell_volume_voxels": float(np.mean(counts)) if len(counts) else np.nan,
        "std_cell_volume_voxels": float(np.std(counts)) if len(counts) else np.nan,
    })

common_summary = pd.DataFrame(summary_rows)
common_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(common_summary["strain_percent"], common_summary["n_cells"], marker="o")
axes[0].set_xlabel("strain (%)")
axes[0].set_ylabel("number of cells")
axes[0].set_title("Cell count in common region")

axes[1].plot(common_summary["strain_percent"], common_summary["median_cell_volume_voxels"], marker="o", label="median")
axes[1].plot(common_summary["strain_percent"], common_summary["mean_cell_volume_voxels"], marker="o", label="mean")
axes[1].set_xlabel("strain (%)")
axes[1].set_ylabel("cell volume (voxels)")
axes[1].set_title("Cell volume in common region")
axes[1].legend()

plt.tight_layout()
plt.show()

## 11. Overlap-based tracking between consecutive strain steps

This is deliberately conservative. It builds an overlap table between labels at strain \(t\) and labels at \(t+\Delta t\) inside the common mask.

A real tracked cell is not guaranteed to have a one-to-one mapping; deformation, segmentation changes, or registration errors may produce apparent split/merge/birth/death events.

The parameters that matter are:

```python
min_overlap_voxels
min_fraction
```

Increase them to keep only stronger correspondences.


In [ ]:
def overlap_table(labels_a, labels_b, mask, min_overlap_voxels=10):
    a = labels_a[mask].astype(np.int64)
    b = labels_b[mask].astype(np.int64)

    keep = (a > 0) & (b > 0)
    a = a[keep]
    b = b[keep]

    if len(a) == 0:
        return pd.DataFrame(columns=["label_a", "label_b", "overlap_voxels"])

    pairs = np.stack([a, b], axis=1)
    unique_pairs, counts = np.unique(pairs, axis=0, return_counts=True)

    df = pd.DataFrame({
        "label_a": unique_pairs[:, 0].astype(int),
        "label_b": unique_pairs[:, 1].astype(int),
        "overlap_voxels": counts.astype(int),
    })

    df = df[df["overlap_voxels"] >= min_overlap_voxels].copy()
    return df.sort_values("overlap_voxels", ascending=False).reset_index(drop=True)


def classify_pair_tracking(labels_a, labels_b, mask, min_overlap_voxels=10, min_fraction=0.10):
    overlaps = overlap_table(labels_a, labels_b, mask, min_overlap_voxels=min_overlap_voxels)

    a_vals = labels_a[mask]
    b_vals = labels_b[mask]

    size_a = pd.Series(a_vals[a_vals > 0]).value_counts().rename_axis("label_a").rename("size_a")
    size_b = pd.Series(b_vals[b_vals > 0]).value_counts().rename_axis("label_b").rename("size_b")

    overlaps = overlaps.merge(size_a, on="label_a", how="left")
    overlaps = overlaps.merge(size_b, on="label_b", how="left")
    overlaps["frac_of_a"] = overlaps["overlap_voxels"] / overlaps["size_a"]
    overlaps["frac_of_b"] = overlaps["overlap_voxels"] / overlaps["size_b"]

    significant = overlaps[
        (overlaps["frac_of_a"] >= min_fraction) |
        (overlaps["frac_of_b"] >= min_fraction)
    ].copy()

    children_per_parent = significant.groupby("label_a")["label_b"].nunique()
    parents_per_child = significant.groupby("label_b")["label_a"].nunique()

    events = []

    all_a = set(size_a.index.astype(int))
    all_b = set(size_b.index.astype(int))

    tracked_b = set(significant["label_b"].astype(int))

    for lab_a in sorted(all_a):
        n_child = int(children_per_parent.get(lab_a, 0))
        if n_child == 0:
            events.append({
                "event": "death",
                "label_a": lab_a,
                "label_b": 0,
                "n_children": 0,
                "n_parents": np.nan,
            })
        elif n_child == 1:
            child = int(significant.loc[significant["label_a"] == lab_a, "label_b"].iloc[0])
            n_parent = int(parents_per_child.get(child, 0))
            event = "one_to_one" if n_parent == 1 else "merge_candidate"
            events.append({
                "event": event,
                "label_a": lab_a,
                "label_b": child,
                "n_children": n_child,
                "n_parents": n_parent,
            })
        else:
            events.append({
                "event": "split",
                "label_a": lab_a,
                "label_b": 0,
                "n_children": n_child,
                "n_parents": np.nan,
            })

    for lab_b in sorted(all_b - tracked_b):
        events.append({
            "event": "birth",
            "label_a": 0,
            "label_b": lab_b,
            "n_children": np.nan,
            "n_parents": 0,
        })

    return overlaps, significant, pd.DataFrame(events)

In [ ]:
tracking_dir = output_root / "tracking_output_commonz"
tracking_dir.mkdir(exist_ok=True)

min_overlap_voxels = 10
min_fraction = 0.10

tracking_outputs = []
tracking_pairs = []

for i in range(len(aligned) - 1):
    a = aligned[i]
    b = aligned[i + 1]

    overlaps, significant, events = classify_pair_tracking(
        a["labels_common"],
        b["labels_common"],
        common_mask_crop,
        min_overlap_voxels=min_overlap_voxels,
        min_fraction=min_fraction,
    )

    pair_name = f"{i:02d}_{a['strain_percent']}_to_{b['strain_percent']}"
    overlaps.to_csv(tracking_dir / f"{pair_name}_overlaps.csv", index=False)
    significant.to_csv(tracking_dir / f"{pair_name}_significant_overlaps.csv", index=False)
    events.to_csv(tracking_dir / f"{pair_name}_events.csv", index=False)

    summary = events["event"].value_counts().rename_axis("event").rename("count").reset_index()
    summary["from"] = a["strain_percent"]
    summary["to"] = b["strain_percent"]
    tracking_outputs.append(summary)
    tracking_pairs.append({
        "pair_name": pair_name,
        "from_index": i,
        "to_index": i + 1,
        "overlaps": overlaps,
        "significant": significant,
        "events": events,
    })

    print("\n", pair_name)
    print(summary)

tracking_summary = pd.concat(tracking_outputs, ignore_index=True)
tracking_summary.to_csv(tracking_dir / "tracking_event_summary.csv", index=False)
tracking_summary

## 12. Event fractions versus strain

In [ ]:
pivot = tracking_summary.pivot_table(
    index=["from", "to"],
    columns="event",
    values="count",
    fill_value=0,
)

pivot_fraction = pivot.div(pivot.sum(axis=1), axis=0)

ax = pivot_fraction.plot(kind="bar", stacked=True, figsize=(10, 5))
ax.set_ylabel("event fraction")
ax.set_title("Overlap-tracking event fractions")
plt.tight_layout()
plt.show()

pivot, pivot_fraction

## 13. Inspect split / merge candidates

In [ ]:
# Choose transition index to inspect.
transition_index = 0

pair = tracking_pairs[transition_index]
a = aligned[pair["from_index"]]
b = aligned[pair["to_index"]]
events = pair["events"]
significant = pair["significant"]

split_labels = events.loc[events["event"] == "split", "label_a"].astype(int).to_numpy()
print("number of split candidates:", len(split_labels))

if len(split_labels):
    # Pick split candidate with largest total overlap.
    split_scores = []
    for lab in split_labels:
        total_overlap = significant.loc[significant["label_a"] == lab, "overlap_voxels"].sum()
        split_scores.append((lab, total_overlap))
    split_scores = sorted(split_scores, key=lambda x: x[1], reverse=True)

    lab = split_scores[0][0]
    children = significant.loc[significant["label_a"] == lab, "label_b"].astype(int).to_numpy()
    print("parent:", lab, "children:", children)

    parent_mask = a["labels_common"] == lab
    child_mask = np.isin(b["labels_common"], children)

    coords = np.argwhere(parent_mask | child_mask)
    z = int(np.median(coords[:, 0])) if len(coords) else common_mask_crop.shape[0] // 2

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(parent_mask[z], cmap="gray", interpolation="nearest", aspect=2.8)
    axes[0].set_title(f"parent {lab}, z={z}")
    axes[0].axis("off")

    axes[1].imshow(child_mask[z], cmap="gray", interpolation="nearest", aspect=2.8)
    axes[1].set_title("children")
    axes[1].axis("off")

    overlay = np.zeros((*parent_mask[z].shape, 3), dtype=float)
    overlay[..., 0] = parent_mask[z]
    overlay[..., 1] = child_mask[z]
    axes[2].imshow(overlay, interpolation="nearest", aspect=2.8)
    axes[2].set_title("red=parent, green=children")
    axes[2].axis("off")
    plt.tight_layout()
    plt.show()

## 14. Cell statistics in the common region

In [ ]:
def compute_cell_table_from_labels(labels, features, mask):
    label_ids, counts = np.unique(labels[(labels > 0) & mask], return_counts=True)
    n_max = int(label_ids.max()) if len(label_ids) else 0

    medians = np.full((n_max + 1, features.shape[-1]), np.nan, dtype=float)
    rows = []

    for lab, count in zip(label_ids, counts):
        region = (labels == lab) & mask
        vals = features[region]
        vals = vals[np.all(np.isfinite(vals), axis=1)]
        if vals.size == 0:
            continue

        med = np.nanmedian(vals, axis=0)
        medians[int(lab)] = med

        spread = np.linalg.norm(vals - med, axis=1)

        row = {
            "label": int(lab),
            "volume_voxels": int(count),
            "internal_spread_median": float(np.nanmedian(spread)),
            "internal_spread_mean": float(np.nanmean(spread)),
            "internal_spread_q95": float(np.nanpercentile(spread, 95)),
        }
        for c in range(features.shape[-1]):
            row[f"median_c{c}"] = float(med[c])
        rows.append(row)

    return pd.DataFrame(rows), medians


def neighbour_offsets():
    offsets = []
    for dy in [-1, 0, 1]:
        for dx in [-1, 0, 1]:
            if dy == 0 and dx == 0:
                continue
            offsets.append((0, dy, dx))
    offsets.append((-1, 0, 0))
    offsets.append((1, 0, 0))

    unique_offsets = []
    seen = set()
    for dz, dy, dx in offsets:
        a = (dz, dy, dx)
        b = (-dz, -dy, -dx)
        if b in seen:
            continue
        seen.add(a)
        unique_offsets.append(a)
    return unique_offsets


def find_neighbour_pairs(labels, mask):
    Z, Y, X = labels.shape
    pairs = set()

    for dz, dy, dx in neighbour_offsets():
        z0a, z1a = max(0, dz), Z + min(0, dz)
        y0a, y1a = max(0, dy), Y + min(0, dy)
        x0a, x1a = max(0, dx), X + min(0, dx)

        z0b, z1b = max(0, -dz), Z + min(0, -dz)
        y0b, y1b = max(0, -dy), Y + min(0, -dy)
        x0b, x1b = max(0, -dx), X + min(0, -dx)

        a = labels[z0a:z1a, y0a:y1a, x0a:x1a]
        b = labels[z0b:z1b, y0b:y1b, x0b:x1b]
        ma = mask[z0a:z1a, y0a:y1a, x0a:x1a]
        mb = mask[z0b:z1b, y0b:y1b, x0b:x1b]

        contact = (a > 0) & (b > 0) & (a != b) & ma & mb
        if not np.any(contact):
            continue

        aa = a[contact].astype(np.int64)
        bb = b[contact].astype(np.int64)
        lo = np.minimum(aa, bb)
        hi = np.maximum(aa, bb)
        for p in zip(lo, hi):
            pairs.add(p)

    if not pairs:
        return np.zeros((0, 2), dtype=np.int64)
    return np.array(sorted(pairs), dtype=np.int64)


def compute_neighbour_table(labels, features, mask, medians):
    pairs = find_neighbour_pairs(labels, mask)
    rows = []

    for lab_a, lab_b in pairs:
        if lab_a >= medians.shape[0] or lab_b >= medians.shape[0]:
            continue
        fa = medians[lab_a]
        fb = medians[lab_b]
        if not (np.all(np.isfinite(fa)) and np.all(np.isfinite(fb))):
            continue

        dvec = fb - fa
        row = {
            "label_a": int(lab_a),
            "label_b": int(lab_b),
            "misorientation": float(np.linalg.norm(dvec)),
        }
        for c in range(features.shape[-1]):
            row[f"delta_c{c}"] = float(dvec[c])
        rows.append(row)

    return pd.DataFrame(rows)


def fit_chi(mis):
    mis = np.asarray(mis, dtype=float)
    mis = mis[np.isfinite(mis)]
    mis = mis[mis > 0]
    if len(mis) < 10:
        return np.nan, np.nan
    k_hat, _, sigma_hat = stats.chi.fit(mis, floc=0)
    return float(k_hat), float(sigma_hat)

In [ ]:
common_stats = []

for a in aligned:
    cell_df, medians = compute_cell_table_from_labels(
        a["labels_common"],
        a["seg_input_crop"],
        common_mask_crop,
    )
    neigh_df = compute_neighbour_table(
        a["labels_common"],
        a["seg_input_crop"],
        common_mask_crop,
        medians,
    )
    k_hat, sigma_hat = fit_chi(neigh_df["misorientation"].to_numpy() if len(neigh_df) else np.array([]))

    a["common_cell_table"] = cell_df
    a["common_neighbour_table"] = neigh_df

    common_stats.append({
        "dataset": a["name"],
        "strain_percent": a["strain_percent"],
        "n_cells": int(a["labels_common"].max()),
        "n_neighbour_pairs": int(len(neigh_df)),
        "median_cell_volume_voxels": float(cell_df["volume_voxels"].median()),
        "mean_cell_volume_voxels": float(cell_df["volume_voxels"].mean()),
        "chi_k": k_hat,
        "chi_sigma": sigma_hat,
        "mean_neighbour_misorientation": float(neigh_df["misorientation"].mean()) if len(neigh_df) else np.nan,
        "median_neighbour_misorientation": float(neigh_df["misorientation"].median()) if len(neigh_df) else np.nan,
    })

common_stats_df = pd.DataFrame(common_stats)
common_stats_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(common_stats_df["strain_percent"], common_stats_df["chi_k"], marker="o")
axes[0].set_xlabel("strain (%)")
axes[0].set_ylabel(r"$k$")
axes[0].set_title(r"$\chi$ shape parameter")

axes[1].plot(common_stats_df["strain_percent"], common_stats_df["chi_sigma"], marker="o")
axes[1].set_xlabel("strain (%)")
axes[1].set_ylabel(r"$\sigma$")
axes[1].set_title(r"$\chi$ scale parameter")

axes[2].plot(common_stats_df["strain_percent"], common_stats_df["mean_neighbour_misorientation"], marker="o", label="mean")
axes[2].plot(common_stats_df["strain_percent"], common_stats_df["median_neighbour_misorientation"], marker="o", label="median")
axes[2].set_xlabel("strain (%)")
axes[2].set_ylabel("neighbour misorientation")
axes[2].set_title("Neighbour misorientation")
axes[2].legend()

plt.tight_layout()
plt.show()

## 15. Histogram and χ fit for one strain step

In [ ]:
which = -1  # last strain step by default

a = aligned[which]
neigh_df = a["common_neighbour_table"]

mis = neigh_df["misorientation"].to_numpy()
mis = mis[np.isfinite(mis)]
mis = mis[mis > 0]

k_hat, _, sigma_hat = stats.chi.fit(mis, floc=0)

x = np.linspace(0, np.percentile(mis, 99.5), 500)
pdf = stats.chi.pdf(x, df=k_hat, loc=0, scale=sigma_hat)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(mis, bins=80, density=True, alpha=0.65, label="data")
ax.plot(x, pdf, linewidth=2, label=fr"$\chi$ fit, $k={k_hat:.3f}$, $\sigma={sigma_hat:.4f}$")
ax.set_xlabel(r"Neighbour misorientation $\sqrt{\Delta\chi^2 + \Delta\phi^2}$")
ax.set_ylabel("Probability density")
ax.set_title(f"{a['strain_percent']}% neighbour misorientation")
ax.legend()
plt.tight_layout()
plt.show()

qs = np.linspace(0.01, 0.99, 99)
emp_q = np.quantile(mis, qs)
chi_q = stats.chi.ppf(qs, df=k_hat, loc=0, scale=sigma_hat)

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(chi_q, emp_q, s=15)
ax.plot([chi_q.min(), chi_q.max()], [chi_q.min(), chi_q.max()], "k--")
ax.set_xlabel(r"$\chi$ theoretical quantile")
ax.set_ylabel("Empirical quantile")
ax.set_title("Q-Q plot")
plt.tight_layout()
plt.show()

print("k:", k_hat)
print("sigma:", sigma_hat)

## 16. Save aligned cropped arrays and tables

In [ ]:
aligned_dir = output_root / "aligned_common_crop"
aligned_dir.mkdir(exist_ok=True)

np.save(aligned_dir / "common_mask_crop.npy", common_mask_crop)
registration_table.to_csv(aligned_dir / "registration_table.csv", index=False)
mode_comparison.to_csv(aligned_dir / "registration_mode_comparison.csv", index=False)
common_summary.to_csv(aligned_dir / "common_region_summary.csv", index=False)
common_stats_df.to_csv(aligned_dir / "common_region_cell_statistics.csv", index=False)

for a in aligned:
    safe = str(a["strain_percent"]).replace(".", "p")
    np.save(aligned_dir / f"{safe}_labels_common.npy", a["labels_common"])
    np.save(aligned_dir / f"{safe}_registered_crop.npy", a["registered_crop"])
    np.save(aligned_dir / f"{safe}_seg_input_crop.npy", a["seg_input_crop"])
    np.save(aligned_dir / f"{safe}_mask_crop.npy", a["mask_crop"])
    np.save(aligned_dir / f"{safe}_rgb_crop.npy", a["rgb_crop"])

    a["common_cell_table"].to_csv(aligned_dir / f"{safe}_cell_table_common.csv", index=False)
    a["common_neighbour_table"].to_csv(aligned_dir / f"{safe}_neighbour_table_common.csv", index=False)

print("saved:", aligned_dir)